# Autonomous Graph of Thoughts (GoT) Debugging Agent
### Quantized Gemma LLM (`google/gemma-2-2b-it` / `gemma-3-4b-it`) + Condensed Code Property Graph & BLSP Runtime Verification in Google Colab

This notebook runs an autonomous LLM debugging agent powered by **Code Property Graph (CPG)** static analysis, **Language-Agnostic CPG Condensation (`cpg_condenser.py`)**, and **Blind-Locate, Surgical-Patch (`verify_patch`)** runtime verification.

**Core Mechanism: Graph of Thoughts (GoT) + CPG Condensation + BLSP**
- The raw Joern CPG is condensed via `condense_cpg(G)` into statement-level CFG/DDG nodes with precomputed **Node Data-Flow Summaries (`reads`, `writes`, `calls`)** on every `METHOD` node.
- The model **cannot see the whole codebase**.
- Given a failing test symptom, it forms hypotheses, invokes graph traversal tools (`condense_method_slice`, `verify_patch`, `trace_variable_to_sink`, `traverse_ast`, `traverse_cfg`, `traverse_data_flow`, `semantic_code_search`), merges observations across syntax, execution paths, and data dependencies, and applies a surgical patch verified against `pytest`.

## Step 1: Clone Repository directly into Colab
Set your GitHub repository URL below. Colab will clone it and enter the repository folder.

In [ ]:
import os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Repository setup
REPO_URL = "https://github.com/nour-ezzehi/GDAC.git"
REPO_DIR = "/content/GDAC"

if not os.path.exists(REPO_DIR):
    print(f"[*] Cloning {REPO_URL}...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("[*] GDAC directory exists. Fetching latest updates...")

%cd {REPO_DIR}
!git checkout main
!git pull origin main
!git log -1 --oneline


## Step 2: Install GPU Packages
Install PyTorch, HuggingFace Transformers, BitsAndBytes (for 4-bit quantization), Accelerate, NetworkX, Sentence-Transformers, and LangGraph.

In [ ]:
!nvidia-smi
!pip install -q torch transformers bitsandbytes accelerate networkx sentence-transformers langchain langchain-core langgraph pydantic pytest


## Step 3: CPG Condensation Benchmark & Dry-Run Verification
Verify that `cpg_condenser.py` shrinks the CPG and token payloads while preserving 100% of causal bug lines, and run the dry-run agent simulation.

In [ ]:
# 1. Run CPG Condenser optimization benchmark
!python cpg_condenser.py --cpg-path cpg_json

# 2. Run local LangGraph simulation verifying all 7 tools (including BLSP verify_patch) and state transitions
!python agent_runner.py --cpg-path cpg_json --dry-run --framework langchain --target discount


## Step 4: Load 4-bit Quantized Gemma Model (`google/gemma-2-2b-it`)
Loads **Gemma 2 2B Instruct** into GPU VRAM using 4-bit NormalFloat (NF4) quantization (**~1.8 GB VRAM**), which runs fast and leaves over 12 GB of free VRAM on Colab's T4 GPU. Automatically frees any existing model in VRAM if re-run, and automatically falls back to `unsloth/gemma-2-2b-it` if no HuggingFace token is configured.

In [ ]:
import gc, torch
from agent_runner import load_quantized_model

# Safely clear any previously loaded model in VRAM before loading
for _var in ("chat_model", "model", "tokenizer"):
    if _var in globals():
        del globals()[_var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Loads Gemma 2 2B Instruct in 4-bit (~1.8 GB VRAM)
# Other lightweight options: "unsloth/gemma-3-4b-it" (~2.8 GB VRAM) or "google/gemma-4-E2B-it"
model, tokenizer = load_quantized_model(
    model_name="google/gemma-2-2b-it",
    quantization="4bit"
)

## Step 5: Execute Autonomous Graph of Thoughts Debugging
The agent will:
1. Read the test failure report and pre-retrieved Entry Point hints (which include `reads`, `writes`, `calls`, and suggested `target_vars`).
2. Formulate an initial hypothesis and emit a `<tool_call>` for `condense_method_slice`.
3. Inspect only the causal statements affecting the target variables.
4. Call `verify_patch` (BLSP) to materialize the 15-line source window, apply the surgical fix, and verify `returncode == 0` via `pytest`.

### Bug Scenario 1: Discount Calculation Defect (`apply_discount`)

In [ ]:
from cpg_bridge import load_cpg_json
from cpg_condenser import condense_cpg
from agent_prompt import format_user_problem_prompt
from agent_langchain import LocalHuggingFaceChatModel, run_langgraph_agent_loop

raw_graph = load_cpg_json("cpg_json")
graph = condense_cpg(raw_graph)
chat_model = LocalHuggingFaceChatModel(model=model, tokenizer=tokenizer)

problem_prompt_1 = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_inventory.py::TestInventoryManager::test_apply_discount\n"
        "AssertionError: 20.0 != 80.0\n"
        "Applying a 20% discount on $100 item resulted in $20.00 instead of $80.00."
    ),
    bug_description="Unit test fails when calculating percentage discounts.",
)

deduction_1 = run_langgraph_agent_loop(chat_model, graph, problem_prompt_1, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(deduction_1)


### Bug Scenario 2: Category Summary Aggregation Defect (`generate_category_summary`)

In [ ]:
from agent_prompt import format_user_problem_prompt
from agent_langchain import LocalHuggingFaceChatModel, run_langgraph_agent_loop

chat_model = LocalHuggingFaceChatModel(model=model, tokenizer=tokenizer)

problem_prompt_2 = format_user_problem_prompt(
    failing_test_output=(
        "FAILED tests/test_reports.py::TestInventoryReporter::test_generate_category_summary_aggregates_multiple_items\n"
        "AssertionError: 250.0 != 1000.0\n"
        "Category summary only shows the last product value ($250) instead of aggregate total ($1000)."
    ),
    bug_description="Category valuation summary ignores multiple items in the same category.",
)

deduction_2 = run_langgraph_agent_loop(chat_model, graph, problem_prompt_2, max_turns=6)
print("\n================ FINAL AGENT DEDUCTION ================")
print(deduction_2)
